# Make Figures 13 and 14 (SED Fit Results)

### Load necessary packages, create a spectral model grid, and define some functions

In [ ]:
from jax import config
config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.signal import convolve as jax_convolve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import corner
import astropy.units as u
import shone
from chromatic import *
from bt_settl import get_interp_stellar_spectrum_3d
import arviz as az

# Create custom binned bin edges
wl_min, wl_max = 0.7, 1.7
n_points = 10000

binned_bin_edges = np.geomspace(wl_min, wl_max, n_points + 1)

binned_wavelengths = (binned_bin_edges[:-1] + binned_bin_edges[1:]) / 2.0

print(f"Creating custom binned grid with {n_points} points from {wl_min} to {wl_max} µm")
print(f"Binned bin edges shape: {len(binned_bin_edges)}")
print(f"Binned wavelengths shape: {len(binned_wavelengths)}")

# Step 2: Create the interpolation grid using these custom bin edges
print("\nCreating BT-Settl 3D interpolation grid with custom binning...")
panchromatic_btsettl_grid = get_interp_stellar_spectrum_3d(binned_bin_edges)
panchromatic_wavelengths = binned_wavelengths
panchromatic_bin_edges = binned_bin_edges

print(f"\nWavelength range: {panchromatic_wavelengths[0]:.3f} - {panchromatic_wavelengths[-1]:.3f} µm")
print(f"Number of wavelength points: {len(panchromatic_wavelengths)}")
print("\nGrid creation complete!")

In [ ]:
visits = {
    'F21': {
        'Grism': 'G141',
        'BJD_times': np.array(pd.read_csv('../../data/F21_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459455.708 * u.day,
        'time_upper': 2459455.738 * u.day,
        'T0 (BJD_TDB)': 2459455.9895 * u.day,
        'exp (s)': 4.9784 * u.s,
        'native resolution': 46.3 * u.angstrom
    },
    'S22': {
        'Grism': 'G102',
        'BJD_times': np.array(pd.read_csv('../../data/S22_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459684.215 * u.day,
        'time_upper': 2459684.243 * u.day,
        'T0 (BJD_TDB)': 2459684.4959 * u.day, # This is 27 planetary orbits after the first transit, + 0.0054 days (the transit arrived 7 minutes late)
        'exp (s)': 9.67632 * u.s,
        'native resolution': 24.6 * u.angstrom
    }
}

systeminfo = {
    'duration (hr)': 3.5 * u.hr,
    'T_orb (d)': 8.463 * u.day,
    'T_rot (d)': 4.86 * u.day,
    'inclination': 89.5,
    'eccentricity': 0.0,
    'longitude_of_periastron': 88.4
}

def read_sensitivity_curve(grism='G141'):
    path = f'../../data/WFC3.IR.{grism}.1st.sens.2.fits'

    response = fits.open(path)

    w = response[1].data['wavelength']/1e4 * u.micron
    s = response[1].data['sensitivity'] * u.cm * u.cm / u.erg
    e = response[1].data['error'] * u.cm * u.cm / u.erg
    
    return w, s, e

def convolve_spectrum_jax(model_wavelength, model_flux, sigma, kernel_size=7):
    """
    Properly convolve a spectrum with a Gaussian kernel in JAX.
    
    Args:
        model_wavelength: Array of wavelengths
        model_flux: Corresponding flux values
        sigma: Standard deviation of Gaussian kernel in wavelength units
        kernel_size: Number of elements in the kernel
        
    Returns:
        Convolved flux array
    """
    # Ensure inputs are JAX arrays
    model_wavelength = jnp.asarray(model_wavelength)
    model_flux = jnp.asarray(model_flux)
    
    # Create proper Gaussian kernel
    x = jnp.linspace(-(kernel_size//2), kernel_size//2, kernel_size)
    kernel = jnp.exp(-0.5 * (x/sigma)**2)
    kernel = kernel / jnp.sum(kernel)  # normalize
    
    # Perform convolution
    convolved = jax_convolve(model_flux, kernel, mode='same', method='fft')
    
    return convolved

def get_BTSettl_spectrum_jax(T, 
                             metallicity, 
                             grid=panchromatic_btsettl_grid):
    """
    Get BT-Settl spectrum for given temperature, logg, and metallicity.
    
    Parameters:
    T : effective temperature (K)
    metallicity : metallicity ([M/H])
    grid : 3D interpolation function from get_interp_stellar_spectrum_3d
    
    Returns:
    wave_jax : wavelength array (micrometers)
    flux_jax : flux array (erg/cm^2/s/Å)
    """
    
    gridspec = grid(
        jnp.array(T, dtype=jnp.float64),
        jnp.array(metallicity, dtype=jnp.float64)
    )
    
    re_normed_flux = gridspec
    
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(re_normed_flux)
    
    return wave_jax, flux_jax

## Load Data

In [ ]:
###############################
# S22
###############################

visit = 'S22'
'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow

'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )

'define some arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = oot_spec_err * calibrated_mean_spec

' Convert to JAX arrays'
calibrated_mean_spec_G102 = jnp.array(calibrated_mean_spec)
oot_spec_err_G102 = jnp.array(oot_spec_err)
SED_wavelengths_G102 = jnp.array(_SED_wavelengths)

###############################
# F21
###############################
visit = 'F21'

'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow

'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )

'define some arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
_oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = _oot_spec_err * calibrated_mean_spec
' Convert to JAX arrays'
calibrated_mean_spec_G141 = jnp.array(calibrated_mean_spec)
oot_spec_err_G141 = jnp.array(oot_spec_err)
SED_wavelengths_G141 = jnp.array(_SED_wavelengths)

###############################
# PLOT DATA
###############################
plt.figure()
plt.errorbar(SED_wavelengths_G102,calibrated_mean_spec_G102,yerr=oot_spec_err_G102*100,fmt='o',label='G102',ms=1)
plt.errorbar(SED_wavelengths_G141,calibrated_mean_spec_G141,yerr=oot_spec_err_G141*100,fmt='o',label='G141',ms=1)
plt.legend()
plt.show()
plt.clf()

## Define Plotting Function

In [ ]:
def plot_sed_fit_results(model_designation, n_temps):
    
    result = az.InferenceData.from_netcdf(f'../../samples/{model_designation}')
    samples = result.posterior

    # Each param becomes shape (chain*draw, *extra_dims)
    flat_samples = {
        key: np.asarray(samples[key].values).reshape(-1, *samples[key].shape[2:])
        for key in samples.data_vars
    }

    log_likelihoods = flat_samples['log_likelihood'].ravel()   # (chain*draw,)

    median_log_likelihood = float(np.median(log_likelihoods))
    max_likelihood_idx = int(np.argmax(log_likelihoods))

    # Extract MLE parameters (excluding log_likelihood itself)
    max_likelihood_params = {
        key: flat_samples[key][max_likelihood_idx]
        for key in flat_samples
        if key != 'log_likelihood'
    }

    n_total_samples = log_likelihoods.shape[0]
    random_indices = np.random.choice(
        n_total_samples, size=min(200, n_total_samples), replace=False
    )

    # Dynamic temperature extraction (works for 1T, 2T, 3T)
    temps = []
    for i in range(1, n_temps + 1):
        key = f'T{i}'
        if key not in max_likelihood_params:
            raise KeyError(
                f"Expected temperature parameter '{key}' for n_temps={n_temps}, "
                f"but it was not found in the posterior samples. "
                f"Available keys: {sorted(max_likelihood_params.keys())}"
            )
        temps.append(max_likelihood_params[key])
    print(f"\nTemperature components: {[f'{t:.0f}K' for t in temps]}")

    # Filling factors for all samples
    n_samples = n_total_samples
    all_filling_factors = np.zeros((n_samples, n_temps))
    for idx in range(n_samples):
        scales = [1.0] + [
            10**flat_samples[f'log_ff{i}'][idx]
            for i in range(2, n_temps + 1)
        ]
        total_scale = sum(scales)
        all_filling_factors[idx, :] = [scale / total_scale for scale in scales]

    # FF statistics
    ff_medians = np.median(all_filling_factors, axis=0)
    ff_percentiles = np.percentile(all_filling_factors, [16, 84], axis=0)
    ff_means = np.mean(all_filling_factors, axis=0)
    ff_stds = np.std(all_filling_factors, axis=0)

    print("\n" + "="*60)
    print("FILLING FACTORS FOR EACH COMPONENT (from full posterior)")
    print("="*60)
    for i in range(n_temps):
        print(f"Component {i+1} (T = {temps[i]:.0f}K):")
        print(f"  Median = {ff_medians[i]:.4f} ({ff_medians[i]*100:.2f}%)")
        print(f"  Mean ± Std = {ff_means[i]:.4f} ± {ff_stds[i]:.4f}")
        print(f"  16th-84th percentile = [{ff_percentiles[0,i]:.4f}, {ff_percentiles[1,i]:.4f}]")
        print(f"  Range = [{np.min(all_filling_factors[:,i]):.4f}, {np.max(all_filling_factors[:,i]):.4f}]")
    print(f"\nSum of median filling factors: {np.sum(ff_medians):.4f}")
    print("="*60 + "\n")
    
    # Filling Factor plot
    fig_ff, axes_ff = plt.subplots(1, n_temps, figsize=(4*n_temps, 4), dpi=400)
    if n_temps == 1:
        axes_ff = [axes_ff]
    for i in range(n_temps):
        axes_ff[i].hist(all_filling_factors[:, i], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
        axes_ff[i].axvline(ff_medians[i], color='red', linestyle='--', label=f'Median: {ff_medians[i]:.3f}')
        axes_ff[i].axvline(ff_percentiles[0, i], color='orange', linestyle=':', label='16th/84th')
        axes_ff[i].axvline(ff_percentiles[1, i], color='orange', linestyle=':')
        axes_ff[i].set_xlabel(f'Filling Factor - Comp {i+1}')
        axes_ff[i].set_ylabel('Frequency')
        axes_ff[i].legend(fontsize=8)
        axes_ff[i].set_title(f'T{temps[i]:.0f}K')
    plt.suptitle('Filling Factor Distributions')
    plt.tight_layout()
    plt.savefig(f'{model_designation}_filling_factor_distributions.png', dpi=200)
    #-------------------------------------------------------
    
    'If you are  fitting for wavelength shift and stretch parameters'
    # wl_shift_G102 = max_likelihood_params.get('wl_shift_G102', 0.0)
    # wl_shift_G141 = max_likelihood_params.get('wl_shift_G141', 0.0)
    # wl_stretch_G102 = max_likelihood_params.get('wl_stretch_G102', 1.0)
    # wl_stretch_G141 = max_likelihood_params.get('wl_stretch_G141', 1.0)
    sigma_conv = 1.0
    
    'If you want to adjust the observed wavelengths for each grism separately:'
    # adjusted_wavelengths_G102 = (SED_wavelengths_G102 - jnp.median(SED_wavelengths_G102)) * wl_stretch_G102 + jnp.median(SED_wavelengths_G102) + wl_shift_G102
    # adjusted_wavelengths_G141 = (SED_wavelengths_G141 - jnp.median(SED_wavelengths_G141)) * wl_stretch_G141 + jnp.median(SED_wavelengths_G141) + wl_shift_G141
    # combined_SED_wavelengths = jnp.concatenate([adjusted_wavelengths_G102, adjusted_wavelengths_G141])
    
    'Prepare data'
    spec_G102 = jnp.array(calibrated_mean_spec_G102 + jnp.asarray(max_likelihood_params['offset_G102'], dtype=jnp.float64))
    spec_G141 = jnp.array(calibrated_mean_spec_G141)
    _spec_flux_jax = jnp.concatenate([
        spec_G102 + max_likelihood_params['slope_G102'] * (SED_wavelengths_G102 - SED_wavelengths_G102[0]),
        spec_G141 + max_likelihood_params['slope_G141'] * (SED_wavelengths_G141 - SED_wavelengths_G141[0])
    ])
    max_likelihood_spec_flux = _spec_flux_jax / jnp.mean(_spec_flux_jax)
    max_likelihood_spec_err = jnp.concatenate([
        jnp.array(oot_spec_err_G102 * jnp.asarray(10**max_likelihood_params['beta_G102'], dtype=jnp.float64)),
        jnp.array(oot_spec_err_G141 * jnp.asarray(10**max_likelihood_params['beta_G141'], dtype=jnp.float64))
    ])
    combined_SED_wavelengths = jnp.concatenate([SED_wavelengths_G102, SED_wavelengths_G141])
        
    'Get spectra and scales for max likelihood'
    spectra = [get_BTSettl_spectrum_jax(T=Temp, metallicity=0) for Temp in temps]
    scales = [1.0] + [jnp.asarray(10**max_likelihood_params[f'log_ff{i}'], dtype=jnp.float64) for i in range(2, n_temps + 1)]
    
    'Max likelihood filling factors'
    total_scale = sum(scales)
    ml_filling_factors = [scale / total_scale for scale in scales]
    print("\nMaximum Likelihood Filling Factors:")
    for i, ff in enumerate(ml_filling_factors, 1):
        print(f"  Component {i}: {ff:.4f} ({ff*100:.2f}%)")
    
    'Calculate combined flux from components'
    _max_likelihood_flux = sum(scales[i] * spectra[i][1] for i in range(n_temps))
    component_fluxes = []
    for i in range(n_temps):
        comp_flux = scales[i] * spectra[i][1]
        comp_flux_norm = comp_flux / jnp.mean(_max_likelihood_flux)
        comp_binned = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, comp_flux_norm)
        comp_convolved = convolve_spectrum_jax(combined_SED_wavelengths, comp_binned, sigma=sigma_conv)
        component_fluxes.append(comp_convolved / jnp.mean(comp_convolved))
    
    'Max Likelihood Combined spectrum'
    _model_flux = _max_likelihood_flux / jnp.mean(_max_likelihood_flux)
    binned_model_flux = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, _model_flux)
    convolved = convolve_spectrum_jax(combined_SED_wavelengths, binned_model_flux, sigma=sigma_conv)
    max_likelihood_model_flux = convolved / jnp.mean(convolved)

    'Compute model flux'
    def compute_model_flux(params):
        temps = [jnp.asarray(params[f'T{i}'], dtype=jnp.float64) for i in range(1, n_temps + 1)]
        spectra = [get_BTSettl_spectrum_jax(T=Temp, metallicity=0) for Temp in temps]
        
        scales = [1.0] + [jnp.asarray(10**params[f'log_ff{i}'], dtype=jnp.float64) for i in range(2, n_temps + 1)]
        
        _flux = scales[0] * spectra[0][1]
        for i in range(1, n_temps):
            _flux += scales[i] * spectra[i][1]
        _model_flux = _flux / jnp.mean(_flux)

        wl_shift_G102_rand = params.get('wl_shift_G102', 0.0)
        wl_shift_G141_rand = params.get('wl_shift_G141', 0.0)
        wl_stretch_G102_rand = params.get('wl_stretch_G102', 1.0)
        wl_stretch_G141_rand = params.get('wl_stretch_G141', 1.0)
        
        rand_adj_wl_G102 = (SED_wavelengths_G102 - jnp.median(SED_wavelengths_G102)) * wl_stretch_G102_rand + jnp.median(SED_wavelengths_G102) + wl_shift_G102_rand
        rand_adj_wl_G141 = (SED_wavelengths_G141 - jnp.median(SED_wavelengths_G141)) * wl_stretch_G141_rand + jnp.median(SED_wavelengths_G141) + wl_shift_G141_rand
        rand_combined_wl = jnp.concatenate([rand_adj_wl_G102, rand_adj_wl_G141])
        
        _binned_model_flux = shone.bin_spectrum(rand_combined_wl, panchromatic_wavelengths, _model_flux)
        binned_model_flux = convolve_spectrum_jax(rand_combined_wl, _binned_model_flux, sigma=sigma_conv)

        return rand_combined_wl, binned_model_flux / jnp.mean(binned_model_flux)
    
    #-------------------------------------------------------
    # Model + Data Plot
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]}, dpi=400)
    fig.suptitle(f'0.8-1.64 μm SED | {n_temps}T Model', fontsize=17)
    axs[1].set_title('Residuals', fontsize=14)
    # axs[1].set_xlim(0.805, 1.64)
    # axs[1].set_ylim(-4, 4)
    axs[1].set_xlabel(r'Wavelength ($\mu$m)', fontsize=14)
    axs[0].set_ylabel('Relative Flux', fontsize=14)
    axs[1].set_ylabel(r'$\sigma$', fontsize=14)
    'Plot random posterior samples'
    for idx in random_indices:
        random_params = {key: flat_samples[key][idx] for key in samples.keys()}
        random_wavelengths, random_model_flux = compute_model_flux(random_params)
        axs[0].plot(random_wavelengths[2:-5], random_model_flux[2:-5], color='red', alpha=0.05, linewidth=0.5, zorder=50)
    'Plot individual components'
    colors = ['green', 'purple', 'cyan', 'orange'][:n_temps]
    for i in range(n_temps):
        label = f'T = {temps[i]:.0f} K | ff = {ff_means[i]:.2f} ± {ff_stds[i]:.2f}'
        axs[0].plot(combined_SED_wavelengths[2:-5], ff_means[i] * component_fluxes[i][2:-5],
                    color=colors[i], linewidth=2, linestyle='--', label=label, zorder=100, alpha=0.5)
    'Plot Data'
    axs[0].errorbar(combined_SED_wavelengths[2:-5], max_likelihood_spec_flux[2:-5], yerr=max_likelihood_spec_err[2:-5],
                    fmt='o', color='b', ms=1, alpha=0.7, zorder=150, label='Data')
    'Plot Max Likelihood Model'
    axs[0].plot(combined_SED_wavelengths[2:-5], max_likelihood_model_flux[2:-5], color='k', linewidth=1, zorder=200, label='Model')
    axs[0].legend(loc='upper right', fontsize=10)
    'Plot Residuals'
    axs[1].errorbar(combined_SED_wavelengths[2:-5], (max_likelihood_model_flux[2:-5] - max_likelihood_spec_flux[2:-5]) / max_likelihood_spec_err[2:-5],
                   color='k', yerr=[1]*len(combined_SED_wavelengths[2:-5]), zorder=150, ms=1, fmt='o', alpha=0.7)
    axs[1].axhline(0, color='k', zorder=-100)
    plt.tight_layout()
    plt.savefig(f'{model_designation}_max_likelihood_result.png')
    #-------------------------------------------------------
    # Plot the wavelength shifts
    # plt.figure(figsize=(6,4),dpi=400)
    # fig.suptitle(f'Wavelength Shifts', fontsize=17)
    # plt.scatter(combined_SED_wavelengths,(random_wavelengths-combined_SED_wavelengths)/jnp.diff(combined_SED_wavelengths).mean(),alpha=0.4,s=1)
    # plt.xlabel(r'Wavelength ($\mu$m)', fontsize=14)
    # plt.ylabel('(New-original)/binsize', fontsize=14)
    
    # plt.tight_layout()
    # plt.savefig(f'{model_designation}_wavelength_shifts.png')
    # #-------------------------------------------------------
    # # Plot the wavelength shifts
    # plt.figure(figsize=(6,4),dpi=400)
    # fig.suptitle(f'Wavelength Shifts', fontsize=17)
    # plt.plot(random_wavelengths[2:-5],max_likelihood_spec_flux[2:-5],alpha=0.4)
    # plt.plot(combined_SED_wavelengths[2:-5],max_likelihood_spec_flux[2:-5],alpha=0.4)
    # plt.xlabel(r'Wavelength ($\mu$m)', fontsize=14)
    # # plt.ylabel('(New-original)/binsize', fontsize=14)
    
    # plt.tight_layout()
    # plt.savefig(f'{model_designation}_wavelength_shifts.png')
    #-------------------------------------------------------
    # Prepare corner plot data with derived ff parameters
    # This works for 1T, 2T, and 3T models
    
    # Define which variables to include in the corner plot
    # base_var_names = ['beta_G102', 'beta_G141', 'wl_shift_G102', 'wl_shift_G141',
    #                   'wl_stretch_G102', 'wl_stretch_G141', 'slope_G102', 'slope_G141', 'offset_G102']
    # Create dictionary for corner plot data
    corner_data = {}
    
    # Add base parameters
    # for var in base_var_names:
    #     if var in samples:
    #         corner_data[var] = np.array(samples[var])
    
    # Add temperature parameters
    for i in range(1, n_temps + 1):
        corner_data[f'T{i}'] = np.array(flat_samples[f'T{i}'])
    
    # Add ff parameters based on n_temps
    if n_temps == 1:
        pass  # No ff parameters needed for 1T model
    
    elif n_temps == 2:
        # 2T model: ff2 = 10**log_ff2, ff1 = 1.0 - ff2
        ff2 = all_filling_factors[:, 1]  # Component 2 from the earlier calculation
        ff1 = 1.0 - ff2                  # Component 1 derived
        
        corner_data['ff1'] = ff1
        corner_data['ff2'] = ff2
    
    elif n_temps == 3:
        # 3T model: ff2 = 10**log_ff2, ff3 = 10**log_ff3, ff1 = 1.0 - (ff2 + ff3)
        ff2 = all_filling_factors[:, 1]  # Component 2
        ff3 = all_filling_factors[:, 2]  # Component 3
        ff1 = 1.0 - (ff2 + ff3)          # Component 1 derived
        
        corner_data['ff1'] = ff1
        corner_data['ff2'] = ff2
        corner_data['ff3'] = ff3
    
    # Convert to array for corner plot
    var_names_list = list(corner_data.keys())
    samples_array = np.column_stack([corner_data[key] for key in var_names_list])
    
    # Create corner plot
    figure = corner.corner(
        samples_array,
        labels=var_names_list,
        show_titles=True,
        title_fmt='.3f',
        quantiles=[0.16, 0.5, 0.84]
    )
    plt.savefig(f'{model_designation}_corner.png', dpi=400)
    plt.show()
    #-------------------------------------------------------

## Plot Each Model

In [ ]:
model_designations = ['jointvisit_1T_2000_8chains_SED_btsettl_agss-logg4.5-met0.0_2026-05-13',
                      'jointvisit_2T_2000_8chains_SED_btsettl_agss-logg4.5-met0.0_2026-05-13',
                      'jointvisit_3T_2000_8chains_SED_btsettl_agss-logg4.5-met0.0_2026-05-13']
i=1
for model_designation in model_designations:
    plot_sed_fit_results(model_designation, n_temps=i)
    i+=1